# Análise de atraso de notificação e disponibilidade temporal dos dados

## Contexto

O Sentinela já demonstrou capacidade de identificar retrospectivamente períodos de aumento incomum de SRAG quando comparados a uma referência histórica.

Entretanto, essa validação utiliza bases consolidadas posteriormente. Isso significa que um registro atribuído a uma determinada semana epidemiológica pode ter sido notificado, digitado ou revisado somente dias ou semanas depois.

Consequentemente, o backtest retrospectivo tradicional pode utilizar informações que não estavam disponíveis no momento em que o sinal teria sido emitido.

Esse ponto representa atualmente uma das principais limitações metodológicas para avaliar a capacidade de alerta precoce do Sentinela.

## Problema

A série epidemiológica é construída a partir da data de início dos sintomas (`DT_SIN_PRI`).

Porém, para avaliar se o Sentinela seria capaz de emitir um sinal em tempo quase real, é necessário distinguir:

- quando o evento epidemiológico ocorreu;
- quando o registro foi notificado;
- quando o registro passou a estar disponível na base;
- e quando possíveis revisões posteriores foram incorporadas.

Em outras palavras, não basta saber em qual semana o caso pertence epidemiologicamente. Também é necessário estimar em que momento esse caso poderia ter sido observado pelo sistema.

## Objetivo

Investigar se os arquivos do SIVEP-Gripe possuem campos que permitam reconstruir, total ou parcialmente, a disponibilidade temporal dos registros.

A análise busca identificar datas relacionadas a:

- início dos sintomas;
- notificação;
- digitação;
- internação;
- encerramento;
- atualização;
- revisão do registro.

A partir desses campos, pretende-se avaliar a possibilidade de construir um backtest temporal ou pseudo-prospectivo.

## Pergunta principal

> Utilizando apenas os registros que provavelmente estariam disponíveis em cada momento do tempo, o Sentinela ainda identificaria os sinais observados retrospectivamente?

## Etapas desta análise

1. Inspecionar os campos disponíveis nos arquivos históricos do SIVEP-Gripe.
2. Identificar variáveis relacionadas à entrada e atualização dos registros.
3. Verificar a qualidade e completude dessas datas.
4. Calcular o atraso entre início dos sintomas e notificação, quando possível.
5. Caracterizar a distribuição do atraso de notificação.
6. Avaliar se é possível reconstruir snapshots retrospectivos da base.
7. Comparar contagens disponíveis em cada momento com os valores consolidados.
8. Reexecutar a regra do Sentinela em condições temporais mais realistas.
9. Documentar limitações e hipóteses necessárias para essa reconstrução.

## Conceitos

### Data epidemiológica

A data utilizada para posicionar o caso na série temporal permanece sendo:

`DT_SIN_PRI` — data de início dos primeiros sintomas.

### Data de disponibilidade

Será investigada uma variável que represente, de forma direta ou aproximada, quando o registro passou a ser conhecido pelo sistema.

Essa variável poderá ser uma data de notificação, digitação ou outro campo disponível no SIVEP-Gripe.

### Atraso de notificação

Quando houver uma data adequada, o atraso será definido inicialmente como:

`atraso = data de disponibilidade - DT_SIN_PRI`

Esse indicador permitirá estimar quanto tempo, em média, um evento epidemiológico leva para aparecer na base.

## Níveis de validação do Sentinela

O projeto passa a distinguir três níveis de validação:

### 1. Validação retrospectiva

Utiliza a base consolidada atual.

Responde:

> O método identifica períodos que posteriormente sabemos terem sido epidemiologicamente relevantes?

### 2. Validação temporal ou pseudo-prospectiva

Reconstrói, quando possível, apenas os dados que estariam disponíveis até determinada data.

Responde:

> O método ainda identificaria o sinal usando apenas informações disponíveis naquele momento?

### 3. Validação prospectiva

Utiliza snapshots reais coletados periodicamente pelo Sentinela daqui em diante.

Responde:

> O sistema detecta sinais em tempo real antes ou durante a resposta epidemiológica?

## Limitações esperadas

É possível que os arquivos históricos disponíveis atualmente não permitam reconstruir perfeitamente o estado original da base em cada data.

Campos de notificação podem indicar quando o caso foi registrado, mas não necessariamente reproduzir todas as revisões posteriores.

Por esse motivo, qualquer reconstrução retrospectiva deverá ser interpretada como uma aproximação da disponibilidade temporal real dos dados.

## Resultado esperado

Ao final desta etapa, o Sentinela deverá possuir uma avaliação explícita sobre:

- atraso de notificação;
- completude temporal;
- viabilidade de backtest pseudo-prospectivo;
- limitações da reconstrução histórica;
- e estratégia de armazenamento de snapshots reais para validação prospectiva futura.

## 1. Inspeção dos campos temporais disponíveis

Nesta etapa serão identificadas as colunas do SIVEP-Gripe que podem representar momentos diferentes do ciclo de vida do registro.

In [1]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

historical_files = sorted(
    raw_data_dir.glob("INFLUD*.csv")
)

print("Arquivos encontrados:", len(historical_files))

for path in historical_files:
    print(path.name)

Arquivos encontrados: 8
INFLUD19-23-03-2026.csv
INFLUD20-23-03-2026.csv
INFLUD21-23-03-2026.csv
INFLUD22-23-03-2026.csv
INFLUD23-23-03-2026.csv
INFLUD24-23-03-2026.csv
INFLUD25-28-09-2026.csv
INFLUD26-28-09-2026.csv


In [2]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

reference_2025_path = (
    project_root
    / "data"
    / "processed"
    / "srag_mg_historical_reference_2019_2024.csv"
)

assert reference_2025_path.exists(), (
    f"Arquivo não encontrado: {reference_2025_path}"
)

reference_2025 = pd.read_csv(
    reference_2025_path,
    sep=";",
)

print("✓ Referência 2019–2024 carregada.")
print("Semanas:", len(reference_2025))
display(reference_2025.head())

✓ Referência 2019–2024 carregada.
Semanas: 52


,epi_week,historical_min,q25,historical_median,q75,historical_max,historical_years,has_minimum_history
0,1,48,111.00,586.0,2993.75,4618,6,True
1,2,43,105.00,471.5,3472.00,4559,6,True
2,3,31,129.75,427.0,3170.75,4572,6,True
3,4,21,105.25,363.0,2947.25,3803,6,True
4,5,31,130.50,433.5,2613.75,4097,6,True


In [3]:
import pandas as pd

path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

columns_2025 = pd.read_csv(
    path_2025,
    sep=";",
    encoding="latin-1",
    nrows=0,
).columns.tolist()

for column in columns_2025:
    if any(
        term in column.upper()
        for term in [
            "NOT",
            "DIG",
            "ATUAL",
            "ENCERR",
            "SIN_PRI",
        ]
    ):
        print(column)

NU_NOTIFIC
DT_NOTIFIC
SEM_NOT
DT_SIN_PRI
SG_UF_NOT
CO_MUN_NOT
DT_ENCERRA
DT_DIGITA
FADIGA


## 2. Seleção dos campos temporais relevantes

A inspeção do arquivo de 2025 identificou os seguintes campos relacionados ao ciclo de vida do registro:

- `DT_SIN_PRI` — data de início dos primeiros sintomas;
- `DT_NOTIFIC` — data de notificação;
- `DT_DIGITA` — data de digitação do registro;
- `DT_ENCERRA` — data de encerramento;
- `NU_NOTIFIC` — identificador da notificação.

Para a avaliação de alerta precoce, serão inicialmente comparadas duas possíveis definições de disponibilidade:

1. **Data de notificação:** aproxima o momento em que o caso foi formalmente notificado.
2. **Data de digitação:** aproxima o momento em que o registro passou a estar disponível no sistema eletrônico.

A adequação de cada campo será avaliada a partir de completude, coerência temporal e distribuição dos atrasos em relação a `DT_SIN_PRI`.

In [4]:
df_2025_temporal = pd.read_csv(
    path_2025,
    sep=";",
    encoding="latin-1",
    usecols=[
        "NU_NOTIFIC",
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
        "DT_DIGITA",
        "DT_ENCERRA",
    ],
    dtype="string",
    low_memory=False,
)

df_2025_temporal["SG_UF"] = (
    df_2025_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

mg_2025_temporal = df_2025_temporal.loc[
    df_2025_temporal["SG_UF"].eq("MG")
].copy()

date_columns = [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
    "DT_DIGITA",
    "DT_ENCERRA",
]

for column in date_columns:
    mg_2025_temporal[column] = pd.to_datetime(
        mg_2025_temporal[column],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

print("Registros MG:", len(mg_2025_temporal))

display(
    mg_2025_temporal[
        date_columns
    ].notna().mean().mul(100).round(2).rename(
        "completude_%"
    )
)

Registros MG: 45655


DT_SIN_PRI    100.00
DT_NOTIFIC    100.00
DT_DIGITA     100.00
DT_ENCERRA     95.92
Name: completude_%, dtype: float64

In [5]:
mg_2025_temporal["delay_notification_days"] = (
    mg_2025_temporal["DT_NOTIFIC"]
    - mg_2025_temporal["DT_SIN_PRI"]
).dt.days

mg_2025_temporal["delay_entry_days"] = (
    mg_2025_temporal["DT_DIGITA"]
    - mg_2025_temporal["DT_SIN_PRI"]
).dt.days

In [6]:
display(
    mg_2025_temporal[
        [
            "delay_notification_days",
            "delay_entry_days",
        ]
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

,delay_notification_days,delay_entry_days
count,45655.000000,45655.000000
mean,8.226153,22.586880
std,22.972289,32.386074
min,0.000000,-129.000000
25%,2.000000,7.000000
50%,4.000000,13.000000
75%,7.000000,27.000000
90%,12.000000,47.000000
95%,22.000000,68.000000
99%,121.000000,178.000000


In [7]:
for column in [
    "delay_notification_days",
    "delay_entry_days"
]:
    negative = (
        mg_2025_temporal[column] < 0
    ).sum()

    print(
        column,
        "negativos:",
        int(negative),
    )

delay_notification_days negativos: 0
delay_entry_days negativos: 16


## 3. Escolha inicial da variável de disponibilidade temporal

A comparação entre `DT_NOTIFIC` e `DT_DIGITA` mostrou diferenças importantes.

### Atraso entre início dos sintomas e notificação

- Mediana: 4 dias
- P75: 7 dias
- P90: 12 dias
- P95: 22 dias
- Registros com atraso negativo: 0

### Atraso entre início dos sintomas e digitação

- Mediana: 13 dias
- P75: 27 dias
- P90: 47 dias
- P95: 68 dias
- Registros com atraso negativo: 16

### Decisão metodológica inicial

Para o primeiro backtest temporal do Sentinela, `DT_NOTIFIC` será utilizada como proxy principal da data de disponibilidade do caso.

Essa escolha é baseada em três características observadas:

1. ausência de atrasos negativos;
2. menor atraso mediano;
3. comportamento temporal mais coerente com o fluxo esperado de notificação.

`DT_DIGITA` será mantida como variável de comparação e poderá ser utilizada posteriormente em análises de sensibilidade.

Esta escolha não implica que `DT_NOTIFIC` represente perfeitamente o momento em que o registro passou a estar disponível em todas as extrações do sistema. Trata-se de uma aproximação retrospectiva, sujeita às limitações dos dados consolidados.

In [8]:
lags = [0, 7, 14, 21, 28]

completeness_by_lag = []

for lag in lags:
    proportion = (
        mg_2025_temporal["delay_notification_days"] <= lag
    ).mean()

    completeness_by_lag.append(
        {
            "lags_days": lag,
            "completeness": proportion,
        }
    )

completeness_by_lag = pd.DataFrame(
    completeness_by_lag
)

completeness_by_lag["completeness_pct"] = (
    completeness_by_lag["completeness"]
    * 100
)

display(completeness_by_lag)

,lags_days,completeness,completeness_pct
0,0,0.066674,6.667397
1,7,0.803702,80.370168
2,14,0.920315,92.031541
3,21,0.949688,94.968788
4,28,0.959150,95.915015


## 4. Completude dos registros por defasagem

A análise de 2025 mostra que a maior parte dos registros é notificada nos primeiros dias após o início dos sintomas.

A proporção acumulada de casos notificados foi:

| Defasagem | Completude |
|---|---:|
| D+0 | 6,7% |
| D+7 | 80,4% |
| D+14 | 92,0% |
| D+21 | 95,0% |
| D+28 | 95,9% |

### Interpretação

A própria semana epidemiológica ainda apresenta baixa completude quando observada em tempo real.

Entretanto, após sete dias, aproximadamente 80% dos registros já foram notificados. Após 14 dias, a completude supera 92%.

Esse comportamento indica que o Sentinela deve considerar a maturidade temporal dos dados antes de interpretar uma contagem semanal como definitiva.

Os resultados também sugerem que um backtest pseudo-prospectivo deve avaliar diferentes defasagens, principalmente D+7 e D+14.

In [9]:
from datetime import timedelta

week_17_start = pd.Timestamp("2025-04-20")
week_17_end = pd.Timestamp("2025-04-26")

week_17_cases = mg_2025_temporal.loc[
    mg_2025_temporal["DT_SIN_PRI"].between(
        week_17_start,
        week_17_end,
    )
].copy()

print("Total consolidado SE17:", len(week_17_cases))

Total consolidado SE17: 1087


In [10]:
lags = [0, 7, 14, 21, 28]

week_17_snapshots = []

for lag in lags:
    snapshot_date = (
        week_17_end
        + timedelta(days=lag)
    )

    known_cases = (
        week_17_cases["DT_NOTIFIC"]
        <= snapshot_date
    ).sum()

    week_17_snapshots.append(
        {
            "lag_days": lag,
            "snapshot_date": snapshot_date,
            "known_cases": int(known_cases),
            "final_cases": len(week_17_cases),
            "completeness_pct": (
                known_cases
                / len(week_17_cases)
                * 100
            ),
        }
    )

week_17_snapshots = pd.DataFrame(
    week_17_snapshots
)

display(week_17_snapshots)

,lag_days,snapshot_date,known_cases,final_cases,completeness_pct
0,0,2025-04-26,412,1087,37.902484
1,7,2025-05-03,879,1087,80.864765
2,14,2025-05-10,996,1087,91.628335
3,21,2025-05-17,1017,1087,93.560258
4,28,2025-05-24,1026,1087,94.388224


In [11]:
reference_week_17 = reference_2025.loc[
    reference_2025["epi_week"].eq(17)
].iloc[0]

historical_median = float(
    reference_week_17["historical_median"]
)

q75 = float(
    reference_week_17["q75"]
)

threshold = historical_median * 1.5

print("Mediana histórica:", historical_median)
print("Q75:", q75)
print("Limiar 1,5x:", threshold)

Mediana histórica: 695.0
Q75: 1063.5
Limiar 1,5x: 1042.5


In [12]:
week_17_snapshots["above_q75"] = (
    week_17_snapshots["known_cases"]
    > q75
)

week_17_snapshots["above_factor_threshold"] = (
    week_17_snapshots["known_cases"]
    >= threshold
)

week_17_snapshots["would_signal"] = (
    week_17_snapshots["above_q75"]
    & week_17_snapshots["above_factor_threshold"]
)

display(
    week_17_snapshots[
        [
            "lag_days",
            "snapshot_date",
            "known_cases",
            "final_cases",
            "completeness_pct",
            "above_q75",
            "above_factor_threshold",
            "would_signal",
        ]
    ]
)

,lag_days,snapshot_date,known_cases,final_cases,completeness_pct,above_q75,above_factor_threshold,would_signal
0,0,2025-04-26,412,1087,37.902484,False,False,False
1,7,2025-05-03,879,1087,80.864765,False,False,False
2,14,2025-05-10,996,1087,91.628335,False,False,False
3,21,2025-05-17,1017,1087,93.560258,False,False,False
4,28,2025-05-24,1026,1087,94.388224,False,False,False


In [13]:
first_detectable_signal = (
    week_17_snapshots.loc[
        week_17_snapshots["would_signal"]
    ]
    .sort_values("lag_days")
    .head(1)
)

if first_detectable_signal.empty:
    print(
        "A SE 17 não teria gerado sinal "
        "nas defasagens avaliadas."
    )
else:
    row = first_detectable_signal.iloc[0]

    print(
        "Primeiro sinal detectável:"
    )
    print(
        f"D+{int(row['lag_days'])}"
    )
    print(
        "Data do snapshot:",
        row["snapshot_date"].date(),
    )
    print(
        "Casos conhecidos:",
        int(row["known_cases"]),
    )
    print(
        "Completude:",
        f"{row['completeness_pct']:.1f}%",
    )

A SE 17 não teria gerado sinal nas defasagens avaliadas.


## 5. Reconstrução pseudo-prospectiva da SE 17

A SE 17 de 2025 foi classificada como `SIGNAL` na análise retrospectiva consolidada.

Entretanto, ao reconstruir a disponibilidade dos registros utilizando `DT_NOTIFIC` como proxy, a semana não atingiria os critérios do Sentinela nas defasagens avaliadas.

| Snapshot | Casos conhecidos | Completude | Sinal |
|---|---:|---:|---|
| D+0 | 412 | 37,9% | Não |
| D+7 | 879 | 80,9% | Não |
| D+14 | 996 | 91,6% | Não |
| D+21 | 1.017 | 93,6% | Não |
| D+28 | 1.026 | 94,4% | Não |

Referência da SE 17:

- Mediana histórica: 695,0
- Q75: 1.063,5
- Limiar de 1,5× a mediana: 1.042,5
- Valor consolidado final: 1.087

### Interpretação

Embora a SE 17 seja classificada como sinal na base consolidada, os registros disponíveis nas primeiras quatro semanas após seu encerramento ainda não seriam suficientes para ultrapassar simultaneamente os critérios estatísticos utilizados pelo Sentinela.

Esse resultado evidencia a importância de considerar atraso de notificação em avaliações de alerta precoce.

A classificação retrospectiva de uma semana como `SIGNAL` não implica necessariamente que o sinal estaria disponível em tempo quase real.

In [14]:
coverage_2025_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2025_mg_weekly_coverage.csv"
)

assert coverage_2025_path.exists(), (
    f"Arquivo não encontrado: {coverage_2025_path}"
)

coverage_2025 = pd.read_csv(
    coverage_2025_path,
    sep=";",
)

coverage_2025["week_start"] = pd.to_datetime(
    coverage_2025["week_start"]
)

coverage_2025["week_end"] = pd.to_datetime(
    coverage_2025["week_end"]
)

print("✓ Coverage 2025 carregado.")
print("Semanas:", len(coverage_2025))

display(coverage_2025.head())

✓ Coverage 2025 carregado.
Semanas: 53


,epi_week,epi_year,week_start,week_end,record_count,cumulative_record_count,calendar_status,data_status
0,1,2025,2024-12-29,2025-01-04,397,397,encerrada,observado
1,2,2025,2025-01-05,2025-01-11,422,819,encerrada,observado
2,3,2025,2025-01-12,2025-01-18,413,1232,encerrada,observado
3,4,2025,2025-01-19,2025-01-25,412,1644,encerrada,observado
4,5,2025,2025-01-26,2025-02-01,359,2003,encerrada,observado


In [15]:
from datetime import timedelta

signal_weeks = [17, 18, 19, 20, 21, 22]
lags = [0, 7, 14, 21, 28]

pseudo_results = []

for epi_week in signal_weeks:
    coverage_row = coverage_2025.loc[
        coverage_2025["epi_week"].eq(epi_week)
    ].iloc[0]

    week_start = pd.Timestamp(
        coverage_row["week_start"]
    )

    week_end = pd.Timestamp(
        coverage_row["week_end"]
    )

    week_cases = mg_2025_temporal.loc[
        mg_2025_temporal["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    reference_row = reference_2025.loc[
        reference_2025["epi_week"].eq(epi_week)
    ].iloc[0]

    historical_median = float(
        reference_row["historical_median"]
    )

    q75 = float(
        reference_row["q75"]
    )

    threshold = historical_median * 1.5

    for lag in lags:
        snapshot_date = (
            week_end
            + timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        final_cases = len(week_cases)

        above_q75 = known_cases > q75

        above_factor_threshold = (
            known_cases >= threshold
        )

        pseudo_results.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "final_cases": final_cases,
                "completeness_pct": (
                    known_cases
                    / final_cases
                    * 100
                ),
                "historical_median": historical_median,
                "q75": q75,
                "median_threshold": threshold,
                "above_q75": above_q75,
                "above_factor_threshold":
                    above_factor_threshold,
                "would_signal": (
                    above_q75
                    and above_factor_threshold
                ),
            }
        )

pseudo_results = pd.DataFrame(
    pseudo_results
)

display(pseudo_results)

,epi_week,lag_days,snapshot_date,known_cases,final_cases,completeness_pct,historical_median,q75,median_threshold,above_q75,above_factor_threshold,would_signal
0,17,0,2025-04-26,412,1087,37.902484,695.0,1063.50,1042.50,False,False,False
1,17,7,2025-05-03,879,1087,80.864765,695.0,1063.50,1042.50,False,False,False
2,17,14,2025-05-10,996,1087,91.628335,695.0,1063.50,1042.50,False,False,False
3,17,21,2025-05-17,1017,1087,93.560258,695.0,1063.50,1042.50,False,False,False
4,17,28,2025-05-24,1026,1087,94.388224,695.0,1063.50,1042.50,False,False,False
5,18,0,2025-05-03,477,1586,30.075662,744.5,1178.00,1116.75,False,False,False
6,18,7,2025-05-10,1306,1586,82.345523,744.5,1178.00,1116.75,True,True,True
7,18,14,2025-05-17,1444,1586,91.046658,744.5,1178.00,1116.75,True,True,True
8,18,21,2025-05-24,1496,1586,94.325347,744.5,1178.00,1116.75,True,True,True
9,18,28,2025-05-31,1523,1586,96.027743,744.5,1178.00,1116.75,True,True,True


In [16]:
pseudo_signals = (
    pseudo_results.loc[
        pseudo_results["would_signal"]
    ]
    .sort_values(
        [
            "snapshot_date",
            "epi_week",
        ]
    )
    .reset_index(drop=True)
)

display(pseudo_signals)

,epi_week,lag_days,snapshot_date,known_cases,final_cases,completeness_pct,historical_median,q75,median_threshold,above_q75,above_factor_threshold,would_signal
0,18,7,2025-05-10,1306,1586,82.345523,744.5,1178.00,1116.75,True,True,True
1,18,14,2025-05-17,1444,1586,91.046658,744.5,1178.00,1116.75,True,True,True
2,19,7,2025-05-17,1642,1941,84.595569,847.0,1285.25,1270.50,True,True,True
3,18,21,2025-05-24,1496,1586,94.325347,744.5,1178.00,1116.75,True,True,True
4,19,14,2025-05-24,1803,1941,92.890263,847.0,1285.25,1270.50,True,True,True
5,20,7,2025-05-24,1908,2201,86.687869,986.0,1487.75,1479.00,True,True,True
6,18,28,2025-05-31,1523,1586,96.027743,744.5,1178.00,1116.75,True,True,True
7,19,21,2025-05-31,1854,1941,95.517774,847.0,1285.25,1270.50,True,True,True
8,20,14,2025-05-31,2094,2201,95.138573,986.0,1487.75,1479.00,True,True,True
9,21,7,2025-05-31,2098,2404,87.271215,1029.5,1558.00,1544.25,True,True,True


In [17]:
if pseudo_signals.empty:
    print(
        "Nenhuma das SE 17–22 geraria sinal "
        "nas defasagens avaliadas."
    )
else:
    first = pseudo_signals.iloc[0]

    print("Primeiro sinal pseudo-prospectivo:")
    print("Semana:", int(first["epi_week"]))
    print(
        "Defasagem:",
        f"D+{int(first['lag_days'])}",
    )
    print(
        "Data:",
        first["snapshot_date"].date(),
    )
    print(
        "Casos conhecidos:",
        int(first["known_cases"]),
    )
    print(
        "Completude:",
        f"{first['completeness_pct']:.1f}%",
    )

Primeiro sinal pseudo-prospectivo:
Semana: 18
Defasagem: D+7
Data: 2025-05-10
Casos conhecidos: 1306
Completude: 82.3%


## 6. Primeiro sinal pseudo-prospectivo

A reconstrução temporal das semanas epidemiológicas 17–22 mostrou que o primeiro sinal que permaneceria detectável utilizando apenas casos já notificados ocorreria na **SE 18**, com defasagem de sete dias.

### Resultado

- Semana epidemiológica: SE 18
- Encerramento da semana: 03/05/2025
- Snapshot avaliado: 10/05/2025
- Defasagem: D+7
- Casos conhecidos: 1.306
- Casos consolidados: 1.586
- Completude: 82,3%
- Mediana histórica: 744,5
- Q75: 1.178,0
- Limiar de 1,5× a mediana: 1.116,75
- Resultado: `SIGNAL`

A SE 17, embora classificada como `SIGNAL` na base consolidada, não atingiu os critérios nas defasagens de D+0 a D+28.

### Interpretação

O resultado demonstra que o atraso de notificação altera de forma relevante o momento em que um sinal estatístico se torna detectável.

No episódio analisado, a regra atual do Sentinela não teria antecipado as primeiras medidas oficiais de resposta adotadas em Minas Gerais no início de maio de 2025.

Portanto, esta análise não sustenta, neste momento, uma afirmação de capacidade de alerta precoce para esse evento.

Ela demonstra, entretanto, que o método identifica o aumento de forma consistente quando os dados atingem aproximadamente 80% de completude.

Esse achado motiva a próxima etapa metodológica: avaliar técnicas de correção de incompletude e nowcasting sem utilizar informação futura do período avaliado.

## 7. Perfil histórico de atraso de notificação — 2019 a 2024

Para evitar vazamento de informação futura, o modelo de completude utilizado para avaliar 2025 será construído exclusivamente com dados anteriores ao ano avaliado.

Nesta etapa serão utilizados os registros de SRAG de Minas Gerais entre 2019 e 2024.

Para cada registro, será calculado:

`atraso_notificacao = DT_NOTIFIC - DT_SIN_PRI`

O objetivo é estimar qual proporção dos casos tende a estar notificada após diferentes defasagens temporais.

Antes de utilizar uma curva agregada de completude, será verificado se o comportamento do atraso é razoavelmente consistente entre os anos históricos.

Essa análise permitirá construir uma função de completude histórica que poderá ser aplicada ao backtest pseudo-prospectivo de 2025 sem utilizar informações do próprio ano avaliado.

In [18]:
historical_delay_frames = []

for path in historical_files:
    name = path.name.upper()

    year = int(
        f"20{name.split("INFLUD")[1][:2]}"
    )

    if year < 2019 or year > 2024:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    residence_uf = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    mg = df.loc[
        residence_uf.eq("MG")
    ].copy()

    mg["DT_SIN_PRI"] = pd.to_datetime(
        mg["DT_SIN_PRI"],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

    mg["DT_NOTIFIC"] = pd.to_datetime(
        mg["DT_NOTIFIC"],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

    mg["source_year"] = year

    mg["delay_notification_days"] = (
        mg["DT_NOTIFIC"]
        - mg["DT_SIN_PRI"]
    ).dt.days

    historical_delay_frames.append(
        mg[
            [
                "source_year",
                "DT_SIN_PRI",
                "DT_NOTIFIC",
                "delay_notification_days",
            ]
        ]
    )

historical_delay = pd.concat(
    historical_delay_frames,
    ignore_index=True,
)

print(
    "Registros históricos MG:",
    len(historical_delay),
)

print(
    "Anos:",
    sorted(
        historical_delay[
            "source_year"
        ].unique()
    ),
)

Registros históricos MG: 449280
Anos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [19]:
delay_quality = (
    historical_delay
    .groupby("source_year")
    .agg(
        records=("delay_notification_days", "size"),
        valid_delay=("delay_notification_days", "count"),
        negative_delay=(
            "delay_notification_days",
            lambda values: (values < 0).sum(),
        ),
    )
)

delay_quality["completeness_pct"] = (
    delay_quality["valid_delay"]
    / delay_quality["records"]
    * 100
)

display(delay_quality)

,records,valid_delay,negative_delay,completeness_pct
source_year,,,,
2019,4035,4035,0,100.0
2020,113781,113781,0,100.0
2021,209248,209248,0,100.0
2022,69332,69332,0,100.0
2023,25613,25613,0,100.0
2024,27271,27271,0,100.0


In [20]:
print(
    "Atrasos negativos totais:",
    int(
        (
            historical_delay[
                "delay_notification_days"
            ] < 0
        ).sum()
    ),
)

Atrasos negativos totais: 0


In [21]:
historical_delay_valid = (
    historical_delay.loc[
        historical_delay[
            "delay_notification_days"
        ].notna()
        & historical_delay[
            "delay_notification_days"
        ].ge(0)
    ]
    .copy()
)

print(
    "Registros válidos:",
    len(historical_delay_valid),
)

Registros válidos: 449280


In [22]:
delay_summary_by_year = (
    historical_delay_valid
    .groupby("source_year")[
        "delay_notification_days"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)

display(
    delay_summary_by_year[
        [
            "count",
            "mean",
            "25%",
            "50%",
            "75%",
            "90%",
            "95%",
            "max",
        ]
    ]
)

,count,mean,25%,50%,75%,90%,95%,max
source_year,,,,,,,,
2019,4035.0,7.745725,2.0,5.0,8.0,14.0,20.0,753.0
2020,113781.0,7.742294,2.0,5.0,9.0,14.0,18.0,1463.0
2021,209248.0,8.883946,4.0,7.0,10.0,15.0,19.0,1103.0
2022,69332.0,6.473317,2.0,4.0,7.0,11.0,16.0,1097.0
2023,25613.0,8.003475,2.0,3.0,6.0,11.0,19.0,734.0
2024,27271.0,7.388068,2.0,4.0,7.0,12.0,20.0,436.0


In [23]:
lags = [0, 7, 14, 21, 28]

rows = []

for year, group in historical_delay_valid.groupby(
    "source_year"
):
    for lag in lags:
        completeness = (
            group["delay_notification_days"]
            <= lag
        ).mean()

        rows.append(
            {
               "year": year,
                "lag_days": lag,
                "completeness_pct": (
                    completeness * 100
                ), 
            }
        )

completeness_by_year = pd.DataFrame(rows)

display(
    completeness_by_year.pivot(
        index="year",
        columns="lag_days",
        values="completeness_pct",
    ).round(1)
)

lag_days,0,7,14,21,28
year,,,,,
2019,3.5,71.3,91.1,95.7,96.7
2020,6.0,68.1,91.4,96.4,97.5
2021,4.2,55.0,89.7,96.1,97.4
2022,6.7,80.0,93.7,96.8,97.7
2023,7.7,82.1,92.9,95.7,96.5
2024,7.8,80.9,92.5,95.6,96.6


## 8. Seleção da janela histórica para o modelo de completude

A análise do atraso de notificação entre 2019 e 2024 mostrou mudança relevante no comportamento temporal dos registros.

A completude em D+7 variou de:

- 2019: 71,3%
- 2020: 68,1%
- 2021: 55,0%
- 2022: 80,0%
- 2023: 82,1%
- 2024: 80,9%

Os anos de 2022 a 2024 apresentam comportamento mais homogêneo e mais próximo do observado posteriormente em 2025.

Além disso, os volumes de registros são extremamente diferentes entre os anos, especialmente durante 2020 e 2021. Portanto, uma curva calculada diretamente sobre todos os registros históricos daria peso excessivo aos anos de maior volume.

### Decisão metodológica

Para o primeiro modelo de nowcasting de 2025, será utilizada como referência principal uma curva de completude construída a partir de 2022–2024.

Cada ano terá o mesmo peso na estimação da completude, independentemente do número de registros.

A mediana das proporções anuais será utilizada como estimativa principal.

O período completo de 2019–2024 será posteriormente utilizado como análise de sensibilidade.

Essa escolha é feita exclusivamente com informações anteriores a 2025 e, portanto, não introduz vazamento de informação do período avaliado.

In [24]:
recent_years = [2022, 2023, 2024]

recent_delay = historical_delay_valid.loc[
    historical_delay_valid["source_year"].isin(
        recent_years
    )
].copy()

completion_rows = []

for year, group in recent_delay.groupby("source_year"):
    for lag in range(0, 29):
        completeness = (
            group["delay_notification_days"] <= lag
        ).mean()

        completion_rows.append(
            {
                "source_year": year,
                "lag_days": lag,
                "completeness": completeness
            }
        )

recent_completion_by_year = pd.DataFrame(
    completion_rows
)

In [25]:
completion_model = (
    recent_completion_by_year
    .groupby("lag_days", as_index=False)
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model[
    "expected_completeness_pct"
] = (
    completion_model[
        "expected_completeness"
    ] * 100
)

display(
    completion_model.loc[
        completion_model["lag_days"].isin(
            [0, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.076797,0.067357,0.077848,7.679694
7,7,0.808698,0.799660,0.821146,80.869788
14,14,0.929450,0.924865,0.936898,92.944989
21,21,0.956663,0.956107,0.967749,95.666263
28,28,0.965714,0.964784,0.976807,96.571450


In [26]:
reference_week_17 = reference_2025.loc[
    reference_2025["epi_week"].eq(17)
].iloc[0]

median_week_17 = float(
    reference_week_17["historical_median"]
)

q75_week_17 = float(
    reference_week_17["q75"]
)

threshold_week_17 = (
    median_week_17 * 1.5
)

completion_d7 = completion_model.loc[
    completion_model["lag_days"].eq(7),
    "expected_completeness",
].iloc[0]

known_cases_d7 = int(
    week_17_snapshots.loc[
        week_17_snapshots["lag_days"].eq(7),
        "known_cases",
    ].iloc[0]
)

nowcast_d7 = (
    known_cases_d7
    / completion_d7
)

would_signal_nowcast = (
    nowcast_d7 > q75_week_17
    and nowcast_d7 >= threshold_week_17
)

print(
    "Casos conhecidos D+7:",
    known_cases_d7,
)

print(
    "Completude histórica esperada:",
    f"{completion_d7 * 100:.1f}%",
)

print(
    "Nowcast:",
    f"{nowcast_d7:.1f}",
)

print(
    "Mediana histórica:",
    median_week_17,
)

print(
    "Q75:",
    q75_week_17,
)

print(
    "Limiar 1,5x:",
    threshold_week_17,
)

print(
    "Sinal com nowcast:",
    would_signal_nowcast,
)

Casos conhecidos D+7: 879
Completude histórica esperada: 80.9%
Nowcast: 1086.9
Mediana histórica: 695.0
Q75: 1063.5
Limiar 1,5x: 1042.5
Sinal com nowcast: True


In [27]:
daily_nowcast_week_17 = []

for lag in range(0, 29):
    snapshot_date = (
        pd.Timestamp("2025-04-26")
        + pd.Timedelta(days=lag)
    )

    known_cases = int(
        (
            week_17_cases["DT_NOTIFIC"]
            <= snapshot_date
        ).sum()
    )

    completion_row = completion_model.loc[
        completion_model["lag_days"].eq(lag)
    ]

    if completion_row.empty:
        continue

    expected_completeness = float(
        completion_row[
            "expected_completeness"
        ].iloc[0]
    )

    if expected_completeness <= 0:
        continue

    nowcast = (
        known_cases
        / expected_completeness
    )

    would_signal = (
        nowcast > q75_week_17
        and nowcast >= threshold_week_17
    )

    daily_nowcast_week_17.append(
        {
            "lag_days": lag,
            "snapshot_date": snapshot_date,
            "known_cases": known_cases,
            "expected_completeness_pct": (
                expected_completeness * 100
            ),
            "nowcast": nowcast,
            "would_signal": would_signal,
        }
    )

daily_nowcast_week_17 = pd.DataFrame(
    daily_nowcast_week_17
)

display(daily_nowcast_week_17)

,lag_days,snapshot_date,known_cases,expected_completeness_pct,nowcast,would_signal
0,0,2025-04-26,412,7.679694,5364.797153,True
1,1,2025-04-27,483,19.713249,2450.128906,True
2,2,2025-04-28,636,34.186498,1860.383568,True
3,3,2025-04-29,727,48.703751,1492.698163,True
4,4,2025-04-30,800,60.492831,1322.470752,True
5,5,2025-05-01,835,68.904697,1211.818690,True
6,6,2025-05-02,867,74.991749,1156.127182,True
7,7,2025-05-03,879,80.869788,1086.932484,True
8,8,2025-05-04,893,84.382678,1058.274074,False
9,9,2025-05-05,913,86.687244,1053.211474,False


In [28]:
first_nowcast_signal = (
    daily_nowcast_week_17.loc[
        daily_nowcast_week_17[
            "would_signal"
        ]
    ]
    .sort_values("lag_days")
    .head(1)
)

if first_nowcast_signal.empty:
    print(
        "A SE 17 não geraria sinal "
        "com nowcasting até D+28."
    )
else:
    row = first_nowcast_signal.iloc[0]

    print(
        "Primeiro sinal com nowcasting:"
    )
    print(
        "Defasagem:",
        f"D+{int(row['lag_days'])}",
    )
    print(
        "Data:",
        row["snapshot_date"].date(),
    )
    print(
        "Casos conhecidos:",
        int(row["known_cases"]),
    )
    print(
        "Completude esperada:",
        f"{row['expected_completeness_pct']:.1f}%",
    )
    print(
        "Nowcast:",
        f"{row['nowcast']:.1f}",
    )

Primeiro sinal com nowcasting:
Defasagem: D+0
Data: 2025-04-26
Casos conhecidos: 412
Completude esperada: 7.7%
Nowcast: 5364.8


In [29]:
MIN_COMPLETENESS = 0.80

daily_nowcast_week_17["is_mature"] = (
    daily_nowcast_week_17["expected_completeness_pct"]
    >= MIN_COMPLETENESS * 100
)

display(
    daily_nowcast_week_17.loc[
        daily_nowcast_week_17["is_mature"]
    ]
)

,lag_days,snapshot_date,known_cases,expected_completeness_pct,nowcast,would_signal,is_mature
7,7,2025-05-03,879,80.869788,1086.932484,True,True
8,8,2025-05-04,893,84.382678,1058.274074,False,True
9,9,2025-05-05,913,86.687244,1053.211474,False,True
10,10,2025-05-06,943,89.013731,1059.387118,False,True
11,11,2025-05-07,960,90.430641,1061.587082,False,True
12,12,2025-05-08,981,91.273962,1074.786252,True,True
13,13,2025-05-09,993,91.929879,1080.171112,True,True
14,14,2025-05-10,996,92.944989,1071.601613,True,True
15,15,2025-05-11,999,93.718034,1065.963464,True,True
16,16,2025-05-12,1004,94.237301,1065.395534,True,True


In [30]:
first_mature_nowcast_signal = (
    daily_nowcast_week_17.loc[
        daily_nowcast_week_17["is_mature"]
        & daily_nowcast_week_17["would_signal"]
    ]
    .sort_values("lag_days")
    .head(1)
)

if first_mature_nowcast_signal.empty:
    print("Nenhum sinal maduro identificado.")
else:
    row = first_mature_nowcast_signal.iloc[0]

    print("Primeiro sinal maduro com nowcasting:")
    print("Defasagem:", f"D+{int(row['lag_days'])}")
    print("Data:", row["snapshot_date"].date())
    print("Casos conhecidos:", int(row["known_cases"]))
    print(
        "Completude esperada:",
        f"{row['expected_completeness_pct']:.1f}%"
    )
    print("Nowcast:", f"{row['nowcast']:.1f}")

Primeiro sinal maduro com nowcasting:
Defasagem: D+7
Data: 2025-05-03
Casos conhecidos: 879
Completude esperada: 80.9%
Nowcast: 1086.9


## 10. Primeiro sinal maduro com nowcasting

Ao incorporar um modelo histórico de completude baseado nos anos de 2022 a 2024 e exigir maturidade mínima de 80%, a SE 17 passa a produzir um sinal utilizável em D+7.

### Resultado

- Semana epidemiológica: SE 17
- Encerramento da semana: 26/04/2025
- Data da avaliação: 03/05/2025
- Defasagem: D+7
- Casos conhecidos: 879
- Completude histórica esperada: 80,9%
- Nowcast: 1.086,9
- Mediana histórica: 695,0
- Q75: 1.063,5
- Limiar de 1,5× a mediana: 1.042,5
- Resultado: `SIGNAL`

### Interpretação

Sem correção de atraso de notificação, a SE 17 não atingiria os critérios do Sentinela nas defasagens avaliadas.

Com a aplicação do nowcast, utilizando exclusivamente parâmetros estimados a partir de anos anteriores e respeitando uma maturidade mínima de 80%, a semana passa a atingir os critérios em D+7.

Esse resultado reduz substancialmente o atraso observado no backtest pseudo-prospectivo sem correção.

A detecção estimada ocorreria em 03/05/2025, praticamente simultânea às primeiras medidas oficiais adotadas em Minas Gerais no início de maio.

O resultado ainda não demonstra antecipação do evento, mas indica que a correção por atraso de notificação pode aproximar o Sentinela de uma operação de alerta quase em tempo real.

## 11. Nowcast das semanas com sinal — SE 17 a SE 22

Após validar a metodologia na SE 17, o mesmo procedimento será aplicado às demais semanas que apresentaram sinal na análise retrospectiva de 2025.

Para cada semana serão avaliados:

- casos conhecidos em cada defasagem;
- completude histórica esperada;
- nowcast;
- maturidade mínima de 80%;
- critérios de sinal do Sentinela;
- primeira data em que o sinal se torna detectável de forma madura.

O objetivo é verificar se o comportamento observado na SE 17 se mantém nas demais semanas do período.

In [31]:
MIN_COMPLETENESS = 0.80

signal_weeks = [17, 18, 19, 20, 21, 22]

nowcast_results = []

for epi_week in signal_weeks:
    coverage_row = coverage_2025.loc[
        coverage_2025["epi_week"].eq(epi_week)
    ].iloc[0]

    week_start = pd.Timestamp(
        coverage_row["week_start"]
    )

    week_end = pd.Timestamp(
        coverage_row["week_end"]
    )

    week_cases = mg_2025_temporal.loc[
        mg_2025_temporal["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    reference_row = reference_2025.loc[
        reference_2025["epi_week"].eq(epi_week)
    ].iloc[0]

    historical_median = float(
        reference_row["historical_median"]
    )

    q75 = float(
        reference_row["q75"]
    )

    median_threshold = (
        historical_median * 1.5
    )

    for lag in range(0, 29):
        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        completion_row = completion_model.loc[
            completion_model["lag_days"].eq(lag)
        ]

        if completion_row.empty:
            continue
        
        expected_completeness = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        is_mature = (
            expected_completeness
            >= MIN_COMPLETENESS
        )

        nowcast = None
        would_signal = False

        if is_mature:
            nowcast = (
                known_cases
                / expected_completeness
            ) 

            would_signal = (
                nowcast > q75
                and nowcast >= median_threshold
            )

        nowcast_results.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "final_cases": len(week_cases),
                "expected_completeness_pct": (
                    expected_completeness * 100
                ),
                "is_mature": is_mature,
                "nowcast": nowcast,
                "historical_median": historical_median,
                "q75": q75,
                "median_threshold": median_threshold,
                "would_signal": would_signal,
            }
        )

nowcast_results = pd.DataFrame(
    nowcast_results
)

display(
    nowcast_results.loc[
        nowcast_results["is_mature"]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,final_cases,expected_completeness_pct,is_mature,nowcast,historical_median,q75,median_threshold,would_signal
7,17,7,2025-05-03,879,1087,80.869788,True,1086.932484,695.0,1063.50,1042.5,True
8,17,8,2025-05-04,893,1087,84.382678,True,1058.274074,695.0,1063.50,1042.5,False
9,17,9,2025-05-05,913,1087,86.687244,True,1053.211474,695.0,1063.50,1042.5,False
10,17,10,2025-05-06,943,1087,89.013731,True,1059.387118,695.0,1063.50,1042.5,False
11,17,11,2025-05-07,960,1087,90.430641,True,1061.587082,695.0,1063.50,1042.5,False
...,...,...,...,...,...,...,...,...,...,...,...,...
169,22,24,2025-06-24,2096,2148,96.115254,True,2180.715249,1145.0,1777.75,1717.5,True
170,22,25,2025-06-25,2100,2148,96.241429,True,2182.012497,1145.0,1777.75,1717.5,True
171,22,26,2025-06-26,2101,2148,96.340435,True,2180.808092,1145.0,1777.75,1717.5,True
172,22,27,2025-06-27,2102,2148,96.468776,True,2178.943363,1145.0,1777.75,1717.5,True


In [32]:
first_signal_by_week = (
    nowcast_results.loc[
        nowcast_results["is_mature"]
        & nowcast_results["would_signal"]
    ]
    .sort_values(
        [
            "epi_week",
            "lag_days",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "final_cases",
            "expected_completeness_pct",
            "nowcast",
            "q75",
            "median_threshold",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,final_cases,expected_completeness_pct,nowcast,q75,median_threshold
0,17,7,2025-05-03,879,1087,80.869788,1086.932484,1063.50,1042.50
1,18,7,2025-05-10,1306,1586,80.869788,1614.941779,1178.00,1116.75
2,19,7,2025-05-17,1642,1941,80.869788,2030.424503,1285.25,1270.50
3,20,7,2025-05-24,1908,2201,80.869788,2359.348327,1487.75,1479.00
4,21,7,2025-05-31,2098,2404,80.869788,2594.293915,1558.00,1544.25
5,22,7,2025-06-07,1945,2148,80.869788,2405.100889,1777.75,1717.50


In [33]:
for _, row in first_signal_by_week.iterrows():
    print(
        f"SE {int(row['epi_week'])}: "
        f"D+{int(row['lag_days'])} | "
        f"{row['snapshot_date'].date()} | "
        f"conhecidos={int(row['known_cases'])} | "
        f"nowcast={row['nowcast']:.1f} | "
        f"completude={row['expected_completeness_pct']:.1f}%"
    )

SE 17: D+7 | 2025-05-03 | conhecidos=879 | nowcast=1086.9 | completude=80.9%
SE 18: D+7 | 2025-05-10 | conhecidos=1306 | nowcast=1614.9 | completude=80.9%
SE 19: D+7 | 2025-05-17 | conhecidos=1642 | nowcast=2030.4 | completude=80.9%
SE 20: D+7 | 2025-05-24 | conhecidos=1908 | nowcast=2359.3 | completude=80.9%
SE 21: D+7 | 2025-05-31 | conhecidos=2098 | nowcast=2594.3 | completude=80.9%
SE 22: D+7 | 2025-06-07 | conhecidos=1945 | nowcast=2405.1 | completude=80.9%


In [34]:
detected_weeks = set(
    first_signal_by_week["epi_week"]
)

missing_weeks = [
    week
    for week in signal_weeks
    if week not in detected_weeks
]

print(
    "Semanas sem sinal maduro:",
    missing_weeks,
)

Semanas sem sinal maduro: []


## 12. Consistência do nowcast nas semanas de sinal

A aplicação do modelo de completude às SE 17–22 mostrou comportamento consistente.

Todas as seis semanas que apresentaram sinal na análise retrospectiva consolidada também apresentaram sinal maduro com nowcasting em D+7.

| Semana | Primeira avaliação madura | Casos conhecidos | Nowcast | Completude esperada |
|---|---|---:|---:|---:|
| SE 17 | 03/05/2025 | 879 | 1.086,9 | 80,9% |
| SE 18 | 10/05/2025 | 1.306 | 1.614,9 | 80,9% |
| SE 19 | 17/05/2025 | 1.642 | 2.030,4 | 80,9% |
| SE 20 | 24/05/2025 | 1.908 | 2.359,3 | 80,9% |
| SE 21 | 31/05/2025 | 2.098 | 2.594,3 | 80,9% |
| SE 22 | 07/06/2025 | 1.945 | 2.405,1 | 80,9% |

### Interpretação

O comportamento consistente entre as seis semanas sugere que o resultado da SE 17 não representa um evento isolado.

A combinação entre:

- maturidade mínima de 80%;
- correção por atraso de notificação;
- comparação com Q75;
- e limiar de 1,5× a mediana histórica;

preservou os sinais em todas as semanas do período epidêmico analisado.

O primeiro sinal maduro com nowcasting ocorreu em 03/05/2025, referente à SE 17.

Essa data é muito próxima das primeiras medidas oficiais de resposta adotadas em Minas Gerais no início de maio de 2025.

O resultado fortalece a hipótese de que a incorporação explícita do atraso de notificação pode melhorar significativamente a capacidade operacional do Sentinela.

## 13. Persistência entre semanas e nível de alerta operacional

Uma semana isolada acima dos limiares pode representar uma flutuação pontual.

Para aumentar a robustez operacional do Sentinela, será introduzido um segundo nível de decisão baseado em persistência temporal.

### Regras

**SIGNAL**

Uma semana epidemiológica recebe `SIGNAL` quando:

- a completude histórica esperada é de pelo menos 80%;
- o nowcast está acima de Q75;
- e o nowcast é pelo menos 1,5× a mediana histórica.

**ALERT**

Um `ALERT` é gerado quando existem pelo menos duas semanas epidemiológicas consecutivas com `SIGNAL` maduro.

Essa regra busca reduzir a interpretação excessiva de alterações isoladas e privilegiar padrões persistentes de aumento.

In [35]:
weekly_alerts = (
    first_signal_by_week[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "final_cases",
            "expected_completeness_pct",
            "nowcast",
            "historical_median",
            "q75",
            "median_threshold",
        ]
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
)

weekly_alerts["signal_status"] = "SIGNAL"

display(weekly_alerts)

,epi_week,lag_days,snapshot_date,known_cases,final_cases,expected_completeness_pct,nowcast,historical_median,q75,median_threshold,signal_status
0,17,7,2025-05-03,879,1087,80.869788,1086.932484,695.0,1063.50,1042.50,SIGNAL
1,18,7,2025-05-10,1306,1586,80.869788,1614.941779,744.5,1178.00,1116.75,SIGNAL
2,19,7,2025-05-17,1642,1941,80.869788,2030.424503,847.0,1285.25,1270.50,SIGNAL
3,20,7,2025-05-24,1908,2201,80.869788,2359.348327,986.0,1487.75,1479.00,SIGNAL
4,21,7,2025-05-31,2098,2404,80.869788,2594.293915,1029.5,1558.00,1544.25,SIGNAL
5,22,7,2025-06-07,1945,2148,80.869788,2405.100889,1145.0,1777.75,1717.50,SIGNAL


In [36]:
weekly_alerts["previous_epi_week"] = (
    weekly_alerts["epi_week"].shift(1)
)

weekly_alerts["has_previous_consecutive_signal"] = (
    weekly_alerts["previous_epi_week"]
    == weekly_alerts["epi_week"] - 1
)

In [37]:
weekly_alerts["operational_status"] = "SIGNAL"

weekly_alerts.loc[
    weekly_alerts[
        "has_previous_consecutive_signal"
    ],
    "operational_status",
] = "ALERT"

In [38]:
display(
    weekly_alerts[
        [
            "epi_week",
            "snapshot_date",
            "nowcast",
            "signal_status",
            "operational_status",
        ]
    ]
)

,epi_week,snapshot_date,nowcast,signal_status,operational_status
0,17,2025-05-03,1086.932484,SIGNAL,SIGNAL
1,18,2025-05-10,1614.941779,SIGNAL,ALERT
2,19,2025-05-17,2030.424503,SIGNAL,ALERT
3,20,2025-05-24,2359.348327,SIGNAL,ALERT
4,21,2025-05-31,2594.293915,SIGNAL,ALERT
5,22,2025-06-07,2405.100889,SIGNAL,ALERT


In [39]:
first_alert = (
    weekly_alerts.loc[
        weekly_alerts[
            "operational_status"
        ].eq("ALERT")
    ]
    .sort_values("snapshot_date")
    .head(1)
)

if first_alert.empty:
    print(
        "Nenhum ALERT operacional identificado."
    )
else:
    row = first_alert.iloc[0]

    print("Primeiro ALERT operacional:")
    print(
        "Semana:",
        int(row["epi_week"]),
    )
    print(
        "Data:",
        row["snapshot_date"].date(),
    )
    print(
        "Nowcast:",
        f"{row['nowcast']:.1f}",
    )
    print(
        "Completude:",
        f"{row['expected_completeness_pct']:.1f}%",
    )

Primeiro ALERT operacional:
Semana: 18
Data: 2025-05-10
Nowcast: 1614.9
Completude: 80.9%


In [40]:
all_weeks_2025 = coverage_2025[
    coverage_2025["epi_week"].between(1, 52)
]["epi_week"].tolist()

full_year_nowcast_results = []

for epi_week in all_weeks_2025:
    coverage_row = coverage_2025.loc[
        coverage_2025["epi_week"].eq(epi_week)
    ].iloc[0]

    week_start = pd.Timestamp(
        coverage_row["week_start"]
    )

    week_end = pd.Timestamp(
        coverage_row["week_end"]
    )

    week_cases = mg_2025_temporal.loc[
        mg_2025_temporal["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    reference_row = reference_2025.loc[
        reference_2025["epi_week"].eq(epi_week)
    ]

    if reference_row.empty:
        continue

    reference_row = reference_row.iloc[0]

    historical_median = float(
        reference_row["historical_median"]
    )

    q75 = float(
        reference_row["q75"]
    )

    median_threshold = (
        historical_median * 1.5
    )

    for lag in range(0, 29):
        completion_row = completion_model.loc[
            completion_model["lag_days"].eq(lag)
        ]

        if completion_row.empty:
            continue

        expected_completeness = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        if expected_completeness < MIN_COMPLETENESS:
            continue

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        nowcast = (
            known_cases
            / expected_completeness
        )

        would_signal = (
            nowcast > q75
            and nowcast >= median_threshold
        )

        full_year_nowcast_results.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "nowcast": nowcast,
                "q75": q75,
                "median_threshold": median_threshold,
                "would_signal": would_signal,
            }
        )

full_year_nowcast_results = pd.DataFrame(
    full_year_nowcast_results
)

In [41]:
first_signal_2025 = (
    full_year_nowcast_results.loc[
        full_year_nowcast_results["would_signal"]
    ]
    .sort_values(
        [
            "epi_week",
            "lag_days",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(first_signal_2025)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,q75,median_threshold,would_signal
0,17,7,2025-05-03,879,1086.932484,1063.50,1042.50,True
1,18,7,2025-05-10,1306,1614.941779,1178.00,1116.75,True
2,19,7,2025-05-17,1642,2030.424503,1285.25,1270.50,True
3,20,7,2025-05-24,1908,2359.348327,1487.75,1479.00,True
4,21,7,2025-05-31,2098,2594.293915,1558.00,1544.25,True
5,22,7,2025-06-07,1945,2405.100889,1777.75,1717.50,True
6,23,7,2025-06-14,2007,2481.767344,2314.50,1764.75,True


In [42]:
first_signal_2025 = first_signal_2025.sort_values(
    "epi_week"
).reset_index(drop=True)

first_signal_2025["previous_week"] = (
    first_signal_2025["epi_week"].shift(1)
)

first_signal_2025["operational_status"] = (
    "SIGNAL"
)

first_signal_2025.loc[
    first_signal_2025["previous_week"]
    == first_signal_2025["epi_week"] - 1,
    "operational_status",
] = "ALERT"

display(
    first_signal_2025[
        [
            "epi_week",
            "snapshot_date",
            "nowcast",
            "operational_status",
        ]
    ]
)

,epi_week,snapshot_date,nowcast,operational_status
0,17,2025-05-03,1086.932484,SIGNAL
1,18,2025-05-10,1614.941779,ALERT
2,19,2025-05-17,2030.424503,ALERT
3,20,2025-05-24,2359.348327,ALERT
4,21,2025-05-31,2594.293915,ALERT
5,22,2025-06-07,2405.100889,ALERT
6,23,2025-06-14,2481.767344,ALERT


## 14. Validação da regra operacional no ano completo de 2025

A regra de nowcasting foi aplicada a todas as semanas epidemiológicas de 2025 com referência histórica disponível.

Foram considerados:

- maturidade mínima de 80%;
- correção por completude histórica;
- nowcast acima de Q75;
- nowcast igual ou superior a 1,5× a mediana histórica;
- persistência entre semanas consecutivas.

### Resultado

O modelo identificou sinais apenas no período compreendido entre as SE 17 e 23.

| Semana | Primeira avaliação madura | Nowcast | Status |
|---|---|---:|---|
| SE 17 | 03/05/2025 | 1.086,9 | SIGNAL |
| SE 18 | 10/05/2025 | 1.614,9 | ALERT |
| SE 19 | 17/05/2025 | 2.030,4 | ALERT |
| SE 20 | 24/05/2025 | 2.359,3 | ALERT |
| SE 21 | 31/05/2025 | 2.594,3 | ALERT |
| SE 22 | 07/06/2025 | 2.405,1 | ALERT |
| SE 23 | 14/06/2025 | 2.481,8 | ALERT |

Não foram identificados outros períodos do ano com sinal maduro segundo a regra atual.

### Interpretação

O agrupamento dos sinais em um único período contínuo sugere que a regra operacional não está produzindo alertas dispersos ao longo do ano.

O comportamento observado é compatível com um episódio epidemiológico persistente, no qual:

- a primeira alteração é identificada como `SIGNAL`;
- a persistência em semanas subsequentes eleva o estado para `ALERT`;
- o alerta permanece ativo enquanto as semanas consecutivas continuam excedendo os limiares estatísticos.

Esse resultado fortalece a robustez do método para o episódio analisado.

## 15. Estado operacional semanal do Sentinela

Para evitar oscilações excessivas entre estados, o Sentinela passa a utilizar uma regra simples de persistência.

### Entrada em alerta

- primeira semana com sinal maduro → `SIGNAL`
- segunda semana consecutiva com sinal maduro → `ALERT`

### Permanência em alerta

O estado `ALERT` permanece ativo enquanto houver continuidade dos sinais ou enquanto ainda não houver duas semanas consecutivas sem sinal.

### Encerramento do alerta

O alerta é encerrado somente após duas semanas epidemiológicas consecutivas sem sinal.

Essa regra introduz uma histerese simples no sistema, reduzindo mudanças de estado provocadas por oscilações pontuais.

In [43]:
weekly_status_2025 = (
    coverage_2025.loc[
        coverage_2025["epi_week"].between(1, 52),
        ["epi_week"],
    ]
    .copy()
)

signal_week_set = set(
    first_signal_2025["epi_week"]
)

weekly_status_2025["has_signal"] = (
    weekly_status_2025["epi_week"].isin(
        signal_week_set
    )
)

display(weekly_status_2025)

,epi_week,has_signal
0,1,False
1,2,False
2,3,False
3,4,False
4,5,False
5,6,False
6,7,False
7,8,False
8,9,False
9,10,False


In [44]:
states = []

current_state = "NORMAL"

consecutive_signals = 0
consecutive_no_signals = 0

for row in weekly_status_2025.itertuples(index=False):
    epi_week = int(row.epi_week)
    has_signal = bool(row.has_signal)

    if has_signal:
        consecutive_signals += 1
        consecutive_no_signals = 0

        if current_state == "NORMAL":
            if consecutive_signals == 1:
                current_state = "SIGNAL"

        elif current_state == "SIGNAL":
            if consecutive_signals >= 2:
                current_state = "ALERT"

        elif current_state == "ALERT":
            current_state = "ALERT"

    else:
        consecutive_no_signals += 1
        consecutive_signals = 0

        if current_state == "SIGNAL":
            current_state = "NORMAL"

        elif current_state == "ALERT":
            if consecutive_no_signals >= 2:
                current_state = "NORMAL"

    states.append(
        {
            "epi_week": epi_week,
            "has_signal": has_signal,
            "operational_state": current_state,
            "consecutive_signals": consecutive_signals,
            "consecutive_no_signals": consecutive_no_signals,
        }
    )

operational_2025 = pd.DataFrame(states)

display(operational_2025)

,epi_week,has_signal,operational_state,consecutive_signals,consecutive_no_signals
0,1,False,NORMAL,0,1
1,2,False,NORMAL,0,2
2,3,False,NORMAL,0,3
3,4,False,NORMAL,0,4
4,5,False,NORMAL,0,5
5,6,False,NORMAL,0,6
6,7,False,NORMAL,0,7
7,8,False,NORMAL,0,8
8,9,False,NORMAL,0,9
9,10,False,NORMAL,0,10


In [45]:
display(
    operational_2025.loc[
        operational_2025["epi_week"].between(
            14,
            28,
        )
    ]
)

,epi_week,has_signal,operational_state,consecutive_signals,consecutive_no_signals
13,14,False,NORMAL,0,14
14,15,False,NORMAL,0,15
15,16,False,NORMAL,0,16
16,17,True,SIGNAL,1,0
17,18,True,ALERT,2,0
18,19,True,ALERT,3,0
19,20,True,ALERT,4,0
20,21,True,ALERT,5,0
21,22,True,ALERT,6,0
22,23,True,ALERT,7,0


In [46]:
alert_start = operational_2025.loc[
    operational_2025["operational_state"].eq("ALERT")
].head(1)

if not alert_start.empty:
    print(
        "ALERT iniciado na SE",
        int(alert_start.iloc[0]["epi_week"]),
    )

ALERT iniciado na SE 18


In [47]:
alert_rows = operational_2025.loc[
    operational_2025["operational_state"].eq("ALERT")
]

if not alert_rows.empty:
    last_alert_week = int(
        alert_rows["epi_week"].max()
    )

    print(
        "Última semana em ALERT:",
        last_alert_week,
    )

Última semana em ALERT: 24


## 16. Ciclo operacional completo do episódio de 2025

A aplicação da regra de persistência permitiu reconstruir o ciclo operacional completo do episódio identificado em 2025.

### Estados observados

- SE 1–16: `NORMAL`
- SE 17: `SIGNAL`
- SE 18–24: `ALERT`
- SE 25 em diante: `NORMAL`

### Início do alerta

O primeiro `ALERT` ocorreu na SE 18, após duas semanas epidemiológicas consecutivas com sinal maduro.

### Encerramento do alerta

A SE 24 não apresentou sinal, mas o estado permaneceu em `ALERT` devido à regra de histerese.

Na SE 25 ocorreu a segunda semana consecutiva sem sinal, levando o sistema de volta ao estado `NORMAL`.

### Interpretação

A regra de persistência produziu um comportamento operacional estável:

- um aumento isolado gera apenas `SIGNAL`;
- a persistência por duas semanas eleva o estado para `ALERT`;
- uma única semana sem sinal não encerra imediatamente o alerta;
- duas semanas consecutivas sem sinal encerram o estado de alerta.

Esse comportamento reduz oscilações entre estados e fornece uma lógica mais adequada para monitoramento epidemiológico operacional.

In [48]:
nowcast_2025_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2025_mg_nowcast.csv"
)

operational_2025_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2025_mg_operational_status.csv"
)

full_year_nowcast_results.to_csv(
    nowcast_2025_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

operational_2025.to_csv(
    operational_2025_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

print("✓", nowcast_2025_path)
print("✓", operational_2025_path)

✓ /home/matheuspessoa/Sentinela/data/processed/srag_2025_mg_nowcast.csv
✓ /home/matheuspessoa/Sentinela/data/processed/srag_2025_mg_operational_status.csv


## 17. Aplicação operacional do nowcast em 2026

Após validar retrospectivamente a metodologia em 2025, o mesmo procedimento será aplicado ao ano corrente de 2026.

Para evitar vazamento temporal, serão utilizados:

- referência epidemiológica de 2019 a 2025;
- modelo de atraso de notificação estimado exclusivamente com 2023 a 2025;
- registros de 2026 disponíveis até a data da extração;
- maturidade mínima de 80% para aplicação do nowcast.

A data da extração utilizada nesta análise é 28/09/2026.

Semanas que ainda não atingiram a maturidade mínima não serão classificadas como `NORMAL`, pois ausência de maturidade não representa ausência de sinal.

In [49]:
model_years_2026 = [2023, 2024, 2025]

delay_frames_2026_model = []

for path in historical_files:
    name = path.name.upper()

    year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if year not in model_years_2026:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False
    )

    mg = df.loc[
        df["SG_UF"]
        .str.strip()
        .str.upper()
        .eq("MG")
    ].copy()

    mg["DT_SIN_PRI"] = pd.to_datetime(
        mg["DT_SIN_PRI"],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

    mg["DT_NOTIFIC"] = pd.to_datetime(
        mg["DT_NOTIFIC"],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

    mg["source_year"] = year

    mg["delay_notification_days"] = (
        mg["DT_NOTIFIC"]
        - mg["DT_SIN_PRI"]
    ).dt.days

    delay_frames_2026_model.append(
        mg[
            [
                "source_year",
                "DT_SIN_PRI",
                "DT_NOTIFIC",
                "delay_notification_days",
            ]
        ]
    )

delay_model_2026 = pd.concat(
    delay_frames_2026_model,
    ignore_index=True,
)

delay_model_2026 = delay_model_2026.loc[
    delay_model_2026["delay_notification_days"].notna()
    & delay_model_2026["delay_notification_days"].ge(0)
].copy()

print(
    "Registros usados:",
    len(delay_model_2026),
)

print(
    "Anos:",
    sorted(delay_model_2026["source_year"].unique()),
)

Registros usados: 98539
Anos: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [51]:
completion_rows_2026 = []

for year, group in delay_model_2026.groupby(
    "source_year"
):
    for lag in range(0, 9):
        completeness = (
            group["delay_notification_days"]
            <= lag
        ).mean()

        completion_rows_2026.append(
            {
                "source_year": year,
                "lag_days": lag,
                "completeness": completeness,
            }
        )

completion_by_year_2026 = pd.DataFrame(
    completion_rows_2026
)

completion_model_2026 = (
    completion_by_year_2026
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_2026[
    "expected_completeness_pct"
] = (
    completion_model_2026[
        "expected_completeness"
    ] * 1
)

display(
    completion_model_2026.loc[
        completion_model_2026[
            "lag_days"
        ].isin([0, 7, 14, 21, 28])
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.076797,0.066674,0.077848,0.076797
7,7,0.808698,0.803702,0.821146,0.808698


In [54]:
path_2026 = next(
    path
    for path in historical_files
    if "INFLUD26" in path.name.upper()
)

df_2026_temporal = pd.read_csv(
    path_2026,
    sep=";",
    encoding="latin-1",
    usecols=[
        "NU_NOTIFIC",
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_2026_temporal["SG_UF"] = (
    df_2026_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

mg_2026_temporal = df_2026_temporal.loc[
    df_2026_temporal["SG_UF"].eq("MG")
].copy()

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    mg_2026_temporal[column] = pd.to_datetime(
        mg_2026_temporal[column],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

mg_2026_temporal["delay_notification_days"] = (
    mg_2026_temporal["DT_NOTIFIC"]
    - mg_2026_temporal["DT_SIN_PRI"]
).dt.days

print(
    "Registros MG:",
    len(mg_2026_temporal),
)

Registros MG: 29087


In [55]:
coverage_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_mg_weekly_coverage.csv"
)

reference_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_mg_historical_reference_2019_2025.csv"
)

assert coverage_2026_path.exists()
assert reference_2026_path.exists()

coverage_2026 = pd.read_csv(
    coverage_2026_path,
    sep=";",
)

reference_2026 = pd.read_csv(
    reference_2026_path,
    sep=";",
)

coverage_2026["week_start"] = pd.to_datetime(
    coverage_2026["week_start"]
)

coverage_2026["week_end"] = pd.to_datetime(
    coverage_2026["week_end"]
)

print(
    "Coverage:",
    len(coverage_2026),
)

print(
    "Referência:",
    len(reference_2026),
)

Coverage: 52
Referência: 52


In [56]:
AS_OF_DATE_2026 = pd.Timestamp(
    "2026-09-28"
)

MIN_COMPLETENESS = 0.80

In [57]:
observed_2026 = coverage_2026.loc[
    coverage_2026["data_status"].eq("observado")
].copy()

observed_2026["age_days"] = (
    AS_OF_DATE_2026
    - observed_2026["week_end"]
).dt.days

display(
    observed_2026[
        [
            "epi_week",
            "week_end",
            "age_days",
        ]
    ].tail(10)
)

,epi_week,week_end,age_days
28,29,2026-07-25,65
29,30,2026-08-01,58
30,31,2026-08-08,51
31,32,2026-08-15,44
32,33,2026-08-22,37
33,34,2026-08-29,30
34,35,2026-09-05,23
35,36,2026-09-12,16
36,37,2026-09-19,9
37,38,2026-09-26,2


In [58]:
nowcast_2026_rows = []

for coverage_row in observed_2026.itertuples(
    index=False
):
    epi_week = int(coverage_row.epi_week)

    week_start = pd.Timestamp(
        coverage_row.week_start
    )

    week_end = pd.Timestamp(
        coverage_row.week_end
    )

    available_age = int(
        (
            AS_OF_DATE_2026
            - week_end
        ).days
    )

    if available_age < 0:
        continue


    reference_row = reference_2026.loc[
        reference_2026["epi_week"].eq(
            epi_week
        )
    ]

    if reference_row.empty:
        continue

    reference_row = reference_row.iloc[0]

    historical_median = float(
        reference_row["historical_median"]
    )

    q75 = float(
        reference_row["q75"]
    )

    median_threshold = (
        historical_median * 1.5
    )

    week_cases = mg_2026_temporal.loc[
        mg_2026_temporal["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    max_lag = min(
        available_age,
        28,
    )

    for lag in range(0, max_lag + 1):
        completion_row = (
            completion_model_2026.loc[
                completion_model_2026[
                    "lag_days"
                ].eq(lag)
            ]
        )

        if completion_row.empty:
            continue

        expected_completeness = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        if (
            expected_completeness
            < MIN_COMPLETENESS
        ):
            continue

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        nowcast = (
            known_cases
            / expected_completeness
        )

        would_signal = (
            nowcast > q75
            and nowcast >= median_threshold
        )

        nowcast_2026_rows.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "nowcast": nowcast,
                "q75": q75,
                "median_threshold":
                    median_threshold,
                "would_signal": would_signal,
            }
        )

nowcast_2026 = pd.DataFrame(
    nowcast_2026_rows
)

display(nowcast_2026.tail())

,epi_week,lag_days,snapshot_date,known_cases,nowcast,q75,median_threshold,would_signal
69,35,8,2026-09-13,409,484.696637,1330.0,801.0,False
70,36,7,2026-09-19,344,425.375170,1270.0,891.0,False
71,36,8,2026-09-20,349,413.591995,1270.0,891.0,False
72,37,7,2026-09-26,256,316.558266,1248.0,792.0,False
73,37,8,2026-09-27,256,303.379802,1248.0,792.0,False


In [59]:
first_signal_2026 = (
    nowcast_2026.loc[
        nowcast_2026["would_signal"]
    ]
    .sort_values(
        [
            "epi_week",
            "lag_days",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(first_signal_2026)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,q75,median_threshold,would_signal
0,15,7,2026-04-25,931,1151.233382,933.5,1047.0,True
1,16,7,2026-05-02,926,1145.050603,1006.5,1102.5,True


In [60]:
if first_signal_2026.empty:
    print(
        "Nenhum SIGNAL maduro identificado "
        "em 2026 até a data da extração."
    )
else:
    display(
        first_signal_2026[
            [
                "epi_week",
                "lag_days",
                "snapshot_date",
                "known_cases",
                "nowcast",
                "q75",
                "median_threshold",
            ]
        ]
    )

,epi_week,lag_days,snapshot_date,known_cases,nowcast,q75,median_threshold
0,15,7,2026-04-25,931,1151.233382,933.5,1047.0
1,16,7,2026-05-02,926,1145.050603,1006.5,1102.5


## 18. Estado operacional de 2026 com controle de maturidade

A aplicação do nowcast em 2026 identificou sinais maduros nas SE 15 e 16.

Pela regra operacional definida anteriormente:

- SE 15 → `SIGNAL`
- SE 16 → `ALERT`
- uma semana sem sinal não encerra imediatamente o alerta;
- duas semanas consecutivas sem sinal encerram o alerta.

Entretanto, no ano corrente existe uma situação adicional: semanas observadas que ainda não atingiram maturidade suficiente para avaliação.

Essas semanas não devem ser classificadas como `NORMAL`, pois a ausência de sinal ainda pode decorrer da incompletude dos registros.

Por esse motivo, o estado operacional passa a distinguir também semanas não maduras.

A maturidade mínima permanece definida como 80% de completude histórica esperada.

In [61]:
MATURE_LAG = int(
    completion_model_2026.loc[
        completion_model_2026[
            "expected_completeness"
        ].ge(MIN_COMPLETENESS),
        "lag_days",
    ].min()
)

print(
    "Primeiro lag considerado maduro:",
    f"D+{MATURE_LAG}",
)

Primeiro lag considerado maduro: D+7


In [64]:
weekly_status_2026 = (
    observed_2026[
        [
            "epi_week",
            "week_start",
            "week_end",
            "age_days",
        ]
    ]
    .copy()
)

weekly_status_2026["is_mature"] = (
    weekly_status_2026["age_days"]
    >= MATURE_LAG
)

signal_week_set_2026 = set(
    first_signal_2026["epi_week"]
)

weekly_status_2026["has_signal"] = pd.NA

weekly_status_2026.loc[
    weekly_status_2026["is_mature"],
    "has_signal",
] = (
    weekly_status_2026.loc[
        weekly_status_2026["is_mature"],
        "epi_week",
    ]
    .isin(signal_week_set_2026)
)

display(
    weekly_status_2026.tail(12)
)

,epi_week,week_start,week_end,age_days,is_mature,has_signal
26,27,2026-07-05,2026-07-11,79,True,False
27,28,2026-07-12,2026-07-18,72,True,False
28,29,2026-07-19,2026-07-25,65,True,False
29,30,2026-07-26,2026-08-01,58,True,False
30,31,2026-08-02,2026-08-08,51,True,False
31,32,2026-08-09,2026-08-15,44,True,False
32,33,2026-08-16,2026-08-22,37,True,False
33,34,2026-08-23,2026-08-29,30,True,False
34,35,2026-08-30,2026-09-05,23,True,False
35,36,2026-09-06,2026-09-12,16,True,False


In [67]:
states_2026 = []

current_state = "NORMAL"

consecutive_signals = 0
consecutive_no_signals = 0

for row in weekly_status_2026.itertuples(
    index=False
):
    epi_week = int(row.epi_week)
    is_madure = bool(row.is_mature)

    if not is_madure:
        states_2026.append(
            {
                "epi_week": epi_week,
                "is_mature": False,
                "has_signal": None,
                "operational_state": "PENDING",
                "last_stable_state": current_state,
                "consecutive_signals":
                    consecutive_signals,
                "consecutive_no_signals":
                    consecutive_no_signals,
            }
        )

        continue

    has_signal = bool(row.has_signal)

    if has_signal:
        consecutive_signals += 1
        consecutive_no_signals = 0

        if current_state == "NORMAL":
            current_state = "SIGNAL"

        elif current_state == "SIGNAL":
            if consecutive_signals >= 2:
                current_state = "ALERT"

        elif current_state == "ALERT":
            current_state = "ALERT"

    else:
        consecutive_signals = 0
        consecutive_no_signals += 1

        if current_state == "SIGNAL":
            current_state = "NORMAL"

        elif current_state == "ALERT":
            if consecutive_no_signals >= 2:
                current_state = "NORMAL"


    states_2026.append(
        {
            "epi_week": epi_week,
            "is_mature": True,
            "has_signal": has_signal,
            "operational_state": current_state,
            "last_stable_state": current_state,
            "consecutive_signals":
                consecutive_signals,
            "consecutive_no_signals":
                consecutive_no_signals,
        }
    )

operational_2026 =pd.DataFrame(
    states_2026
)

display(
    operational_2026.loc[
        operational_2026[
            "epi_week"
        ].between(12, 20)
    ]
)

,epi_week,is_mature,has_signal,operational_state,last_stable_state,consecutive_signals,consecutive_no_signals
11,12,True,False,NORMAL,NORMAL,0,12
12,13,True,False,NORMAL,NORMAL,0,13
13,14,True,False,NORMAL,NORMAL,0,14
14,15,True,True,SIGNAL,SIGNAL,1,0
15,16,True,True,ALERT,ALERT,2,0
16,17,True,False,ALERT,ALERT,0,1
17,18,True,False,NORMAL,NORMAL,0,2
18,19,True,False,NORMAL,NORMAL,0,3
19,20,True,False,NORMAL,NORMAL,0,4


In [68]:
display(
    operational_2026.tail(10)
)

,epi_week,is_mature,has_signal,operational_state,last_stable_state,consecutive_signals,consecutive_no_signals
28,29,True,False,NORMAL,NORMAL,0,13
29,30,True,False,NORMAL,NORMAL,0,14
30,31,True,False,NORMAL,NORMAL,0,15
31,32,True,False,NORMAL,NORMAL,0,16
32,33,True,False,NORMAL,NORMAL,0,17
33,34,True,False,NORMAL,NORMAL,0,18
34,35,True,False,NORMAL,NORMAL,0,19
35,36,True,False,NORMAL,NORMAL,0,20
36,37,True,False,NORMAL,NORMAL,0,21
37,38,False,None,PENDING,NORMAL,0,21


In [70]:
signal_start_2026 = operational_2026.loc[
    operational_2026[
        "operational_state"
    ].eq("SIGNAL")
].head(1)

alert_start_2026 = operational_2026.loc[
    operational_2026[
        "operational_state"
    ].eq("ALERT")
].head(1)

pending_2026 = operational_2026.loc[
    operational_2026[
        "operational_state"
    ].eq("PENDING")
]

if not signal_start_2026.empty:
    print(
        "Primeiro SIGNAL:",
        "SE",
        int(
            signal_start_2026.iloc[0][
                "epi_week"
            ]
        )
    )

if not alert_start_2026.empty:
    print(
        "Primeiro ALERT:",
        "SE",
        int(
            alert_start_2026.iloc[0][
                "epi_week"
            ]
        ),
    )

if not pending_2026.empty:
    print(
        "Semanas ainda imaturas:",
        pending_2026[
            "epi_week"
        ].tolist(),
    )

Primeiro SIGNAL: SE 15
Primeiro ALERT: SE 16
Semanas ainda imaturas: [38]


In [71]:
nowcast_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_mg_nowcast.csv"
)

operational_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_mg_operational_status.csv"
)

nowcast_2026.to_csv(
    nowcast_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

operational_2026.to_csv(
    operational_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

print("✓", nowcast_2026_path)
print("✓", operational_2026_path)

✓ /home/matheuspessoa/Sentinela/data/processed/srag_2026_mg_nowcast.csv
✓ /home/matheuspessoa/Sentinela/data/processed/srag_2026_mg_operational_status.csv
